[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/hamilton-certified/notebooks/day-04-feature-pipeline-part1.ipynb#scrollTo=cc000001)

---
# Day 4 · Feature Engineering Pipeline — Part 1
**certified-journeys / hamilton-certified** · Day 4 · Feature Engineering

> **Goal for today:** Build a complete Hamilton feature pipeline for a tabular ML dataset — 10+ features, tagged by type, collected into a single feature DataFrame ready for a model.

In [ ]:
%pip install -q sf-hamilton

## The Dataset

We'll use a synthetic customer churn dataset: each row is a customer with demographic and behavioural attributes. The goal is to engineer features for a churn prediction model.

**Raw columns:** `age`, `tenure_months`, `monthly_spend`, `num_products`, `has_complaint`, `days_since_last_login`

**Feature engineering goals:**
1. Numerical features — normalized/scaled versions
2. Boolean features — thresholded flags
3. Interaction features — products of two signals
4. Bucketed features — discretized continuous values

We'll build each as a separate Hamilton function — one function, one feature, one node.

In [ ]:
import numpy as np
import pandas as pd

# Generate synthetic dataset — free alternative to a CSV download
rng = np.random.default_rng(42)
N = 500

raw_df = pd.DataFrame({
    'age':                  rng.integers(18, 75, N).astype(float),
    'tenure_months':        rng.integers(0, 72, N).astype(float),
    'monthly_spend':        rng.exponential(80, N),
    'num_products':         rng.integers(1, 6, N).astype(float),
    'has_complaint':        rng.integers(0, 2, N).astype(float),
    'days_since_last_login': rng.integers(0, 90, N).astype(float),
})

print(raw_df.shape)
raw_df.head()

## Step 1 · Raw Input Nodes

The first layer of the pipeline extracts each raw column as a named Series — one function per column. This uses `@extract_columns`, which we saw in Day 3.

**Design principle:** Never reference `df['column']` directly in feature functions. Use named Series parameters so each feature has a clean, testable interface.

In [ ]:
import sys, types
from hamilton import driver
from hamilton.function_modifiers import tag, extract_columns
from hamilton.plugins import h_pandas

# --- raw_features.py ---

@extract_columns(
    'age', 'tenure_months', 'monthly_spend',
    'num_products', 'has_complaint', 'days_since_last_login'
)
def raw_features(raw_data: pd.DataFrame) -> pd.DataFrame:
    """Extract all raw columns from the source DataFrame."""
    return raw_data[[
        'age', 'tenure_months', 'monthly_spend',
        'num_products', 'has_complaint', 'days_since_last_login'
    ]]

print('Raw input layer defined.')

## Step 2 · Numerical Features

Numerical features transform continuous inputs into model-ready values. Each transformation is:
- A single function
- Tagged with `feature_type='numerical'`
- Depends on raw Series by name — no DataFrame lookup

Z-score normalization: `(x - mean) / std`  
Min-max scaling: `(x - min) / (max - min)`

In [ ]:
# --- numerical_features.py ---

@tag(feature_type='numerical', owner='ml-team')
def age_zscore(age: pd.Series) -> pd.Series:
    """Z-score normalized age."""
    return (age - age.mean()) / age.std()

@tag(feature_type='numerical', owner='ml-team')
def tenure_years(tenure_months: pd.Series) -> pd.Series:
    """Customer tenure converted from months to years."""
    return tenure_months / 12.0

@tag(feature_type='numerical', owner='ml-team')
def monthly_spend_log(monthly_spend: pd.Series) -> pd.Series:
    """Log-transformed monthly spend — reduces right skew."""
    return np.log1p(monthly_spend)

@tag(feature_type='numerical', owner='ml-team')
def recency_score(days_since_last_login: pd.Series) -> pd.Series:
    """Recency: inverse of days since login, scaled 0-1. Higher = more recent."""
    max_days = days_since_last_login.max()
    return 1.0 - (days_since_last_login / max_days)

print('Numerical features defined: age_zscore, tenure_years, monthly_spend_log, recency_score')

## Step 3 · Boolean & Categorical Features

Boolean features are thresholded signals — they answer a yes/no question about a customer. Keep the threshold logic inside the function and name the node for the question it answers.

In [ ]:
# --- boolean_features.py ---

@tag(feature_type='boolean', owner='ml-team')
def is_high_spender(monthly_spend: pd.Series) -> pd.Series:
    """True if spend is above the 75th percentile."""
    return (monthly_spend > monthly_spend.quantile(0.75)).astype(float)

@tag(feature_type='boolean', owner='ml-team')
def is_new_customer(tenure_months: pd.Series) -> pd.Series:
    """True if tenure is under 6 months."""
    return (tenure_months < 6).astype(float)

@tag(feature_type='boolean', owner='ml-team')
def is_at_risk(has_complaint: pd.Series, recency_score: pd.Series) -> pd.Series:
    """At-risk: has a complaint AND low recency (not logging in)."""
    return ((has_complaint == 1) & (recency_score < 0.3)).astype(float)

@tag(feature_type='categorical', owner='ml-team')
def product_tier(num_products: pd.Series) -> pd.Series:
    """Bucket num_products into tiers: low(1), mid(2-3), high(4+)."""
    return pd.cut(
        num_products, bins=[0, 1, 3, 6],
        labels=['low', 'mid', 'high']
    ).astype(str)

print('Boolean/categorical features defined: is_high_spender, is_new_customer, is_at_risk, product_tier')

## Step 4 · Interaction Features

Interaction features combine two signals — they capture joint effects that neither feature can express alone. In Hamilton, the dependency is explicit from the parameter names.

**`spend_x_tenure`**: high-spend long-tenured customers behave differently from high-spend new customers.

In [ ]:
# --- interaction_features.py ---

@tag(feature_type='numerical', owner='ml-team')
def spend_x_tenure(monthly_spend_log: pd.Series, tenure_years: pd.Series) -> pd.Series:
    """Interaction: log-spend × tenure years. High = loyal high-value customer."""
    return monthly_spend_log * tenure_years

@tag(feature_type='numerical', owner='ml-team')
def engagement_score(recency_score: pd.Series, num_products: pd.Series) -> pd.Series:
    """Engagement: recency × product breadth — recent multi-product users are most engaged."""
    return recency_score * (num_products / num_products.max())

print('Interaction features defined: spend_x_tenure, engagement_score')

## Step 5 · Assemble & Execute the Full Pipeline

Now we wire everything together with the Driver. All functions defined above are registered as a single module. The `PandasDataFrameResult` adapter collects all outputs into a feature DataFrame.

In [ ]:
# Assemble all functions into one module
feature_pipeline = types.ModuleType('feature_pipeline')
all_fns = [
    raw_features,
    age_zscore, tenure_years, monthly_spend_log, recency_score,
    is_high_spender, is_new_customer, is_at_risk, product_tier,
    spend_x_tenure, engagement_score,
]
for fn in all_fns:
    setattr(feature_pipeline, fn.__name__, fn)
sys.modules['feature_pipeline'] = feature_pipeline

# Build the driver with DataFrame result adapter
dr = (
    driver.Builder()
    .with_modules(feature_pipeline)
    .with_adapters(h_pandas.PandasDataFrameResult())
    .build()
)

print('Pipeline nodes:', [v.name for v in dr.list_available_variables()])

In [ ]:
# Define which features go into the model
MODEL_FEATURES = [
    'age_zscore', 'tenure_years', 'monthly_spend_log', 'recency_score',
    'is_high_spender', 'is_new_customer', 'is_at_risk',
    'spend_x_tenure', 'engagement_score',
]

# Execute — produces a DataFrame
feature_df = dr.execute(MODEL_FEATURES, inputs={'raw_data': raw_df})

print('Feature matrix shape:', feature_df.shape)
print('\nFeature matrix (first 5 rows):')
print(feature_df.head().round(3).to_string())

### What just happened?
- **The Driver executed 10 functions** to produce 9 output features, automatically resolving dependencies.
- **`PandasDataFrameResult`** assembled the individual Series outputs into a single DataFrame — no manual `pd.concat` needed.
- **`is_at_risk`** depends on both `has_complaint` (raw) and `recency_score` (derived) — Hamilton resolved that automatically from parameter names.

## Step 6 · Querying Features by Tag

Now that features are tagged, we can query which nodes have `feature_type='numerical'` vs `'boolean'` — useful for building feature registries or applying different scalers to different types.

In [ ]:
# Query features by type
numerical_features = []
boolean_features = []
categorical_features = []

for v in dr.list_available_variables():
    ft = v.tags.get('feature_type')
    if ft == 'numerical':   numerical_features.append(v.name)
    elif ft == 'boolean':   boolean_features.append(v.name)
    elif ft == 'categorical': categorical_features.append(v.name)

print('Numerical features: ', numerical_features)
print('Boolean features:   ', boolean_features)
print('Categorical features:', categorical_features)

# Execute only numerical features
num_df = dr.execute(numerical_features, inputs={'raw_data': raw_df})
print(f'\nNumerical-only matrix: {num_df.shape}')
print(num_df.describe().round(3))

### What just happened?
- **Tag-based feature selection** lets you build the subset dynamically — no hardcoded lists that go stale.
- You can pass `numerical_features` directly to `execute()` — Hamilton computes only what's needed.
- This pattern scales: add a new `@tag(feature_type='numerical')` function and it's automatically included.

## Step 7 · Unit Testing Individual Features

Every Hamilton function is a pure Python function — test it directly without a Driver.

In [ ]:
# Unit tests — no Driver, no fixtures
test_spend = pd.Series([10.0, 50.0, 200.0, 300.0])

# Test is_high_spender: 75th percentile = 162.5, so 200 and 300 should be True
result = is_high_spender(test_spend)
assert result.tolist() == [0.0, 0.0, 1.0, 1.0], f'Expected [0,0,1,1], got {result.tolist()}'
print('✓ is_high_spender')

# Test monthly_spend_log: log1p is monotonic, and log1p(0)=0
test_spend_with_zero = pd.Series([0.0, 1.0, np.e - 1])
log_result = monthly_spend_log(test_spend_with_zero)
assert abs(log_result.iloc[0]) < 1e-10, 'log1p(0) should be 0'
assert abs(log_result.iloc[1] - np.log(2)) < 1e-10, 'log1p(1) should be log(2)'
print('✓ monthly_spend_log')

# Test tenure_years: 12 months = 1 year
test_tenure = pd.Series([0.0, 12.0, 24.0, 6.0])
years = tenure_years(test_tenure)
assert years.tolist() == [0.0, 1.0, 2.0, 0.5], f'Got {years.tolist()}'
print('✓ tenure_years')

# Test is_at_risk: complaint=1 AND recency < 0.3
complaints = pd.Series([1.0, 1.0, 0.0, 1.0])
recency    = pd.Series([0.1, 0.8, 0.1, 0.2])
at_risk = is_at_risk(complaints, recency)
assert at_risk.tolist() == [1.0, 0.0, 0.0, 1.0], f'Got {at_risk.tolist()}'
print('✓ is_at_risk')

print('\nAll unit tests passed!')

### What just happened?
- **No mocking, no Driver** — each function is tested as a plain Python function with synthetic inputs.
- **Tests caught the exact threshold logic** — is `>=` or `>` correct for `is_high_spender`? The test makes it explicit.
- **`is_at_risk` uses two parameters** — the test supplies both independently, isolating this function from `recency_score`'s logic.

In [ ]:
# Challenge: add a new feature `spend_per_product` (monthly_spend / num_products)
# Tag it as feature_type='numerical'
# Write a unit test for it
# Add it to MODEL_FEATURES and re-run the pipeline

# @tag(feature_type='numerical', owner='ml-team')
# def spend_per_product(monthly_spend: pd.Series, num_products: pd.Series) -> pd.Series:
#     ...

print('Implement spend_per_product, test it, and add it to the pipeline!')

---
## Day 4 key concepts recap

| Concept | What to remember |
|---|---|
| One function = one feature | Split multi-step logic; each node should be independently testable |
| `@tag(feature_type=...)` | Enables tag-based feature selection at execute() time |
| `@extract_columns` | First layer: DataFrame → named Series nodes |
| `PandasDataFrameResult` | Collects all output Series into one DataFrame automatically |
| Unit testing | Call each function directly with synthetic Series — no Driver needed |
| Interaction features | Parameter names express the dependency explicitly in the DAG |

> **Tip:** Keep each function to one transformation. If you find yourself writing two steps in one function, split it — your tests will thank you.

---
## What's next
**Day 5** → Modularity: split your pipeline into multiple Python files, compose them with `.with_modules()`, and use subdag for namespaced sub-pipelines.

Mark Day 4 complete in your [tracker](../index.html).